---
title: Week 5 Tutorial 1, Orchestra Simulation
subtitle: Orchestra Scenario, Fluorite & Gypsum simulation
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-07
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Orchestra as a tool
This example illustrates how to use Python and ORCHESTRA to simulate the Ca - F - Gypsum example from Appelo and Postma chapter 4.
The next python cells imports the necessary libararies and checks the path to the input files required for the Orchestra simulation. Please note that this is only required for the jupyter-book version. For a stand-alone version of this notebook you need to start jupyter-lab from the directory with the input files.

In [1]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

orchestra_path = "." 
# print(orchestra_path)

## Define chemical system: Calcite, Siderite, water using the ORCHESTRA-GUI
In order to solve a chemical equilibrium problem with pyOrchestra, we need to define our chemical system first. Orchestra defines this system with a number of text files. The most important one is the so-called chemistry input file (*chemistry1.inp*). This file is most easily made using the Orchestra GUI which can be accessed by clicking on the orchestrat2023.jar file.


Calcite is $\text{CaCO}_3\text(s)$ and Siderite $\text{FeCO}_3\text(s)$. The master species for this problem are chosed as $\text{Ca}^{+2}\text(aq)$, $\text{Fe}^{+2}\text(aq)$ and $\text{CO}_2\text{g}$. Where amounts are given as Ca+2.tot (phase: diss), Fe+2.tot (phase: diss) and CO2[g].logact (phase: gas) where CO2[g] is provided as a fixed parameter. We also have the water equilibrium so we define H+ (phase: diss and input variable: pH) with a fixed log-activity and H+.logact = -pH. Finally we define H2O as our background solute (liter phase) with a fixed log activity = 0.

The master species table looks like

|Primary entity|Phase| Input Variable|Fix log activity|Log activity|Concentration|Phase|Expression|
|--|--|--|--|--|--|--|--|
|CO2[g]|gas||x|0.0||||||
|Ca+2|diss||||5.0|tot||
|Fe+2|diss||||5.0|tot||
|H+|diss|pH|x|-7.0|||H.logact=-pH|
|H2O|liter||x|0.0||||


Orchestra will use these primary entities, and their initial values to calculate the total elemental composition in the complete system. The fixed log-activties indicate that the total can vary during the simulation, as long as the log-activity condition is maintained. We assume that Orchestra uses the pH to maintain the chargebalance. 

### Phases & Reactions
On the **Phases & Reactions** tab shown in figure [](#phases_reactions) we define the reaction network of our system. The schematic on the left-hand side is used to define how the different phases (gas, liquid and solids) are connected, which we will ignore for now.

The table on this tab shows all possible reactions in this system. You should recognize that this table originates from the log-transformed mass action law. The table shows the logK-value, in which phase the species can be found, followed by the stoichiometry of the reaction as a function of the master-species or convenient secondary species in the system. 

For this example we need to select Calcite[s] and Siderite[s], because we want these minerals to precipitate, Orchestra will force the SI to be zero if the Ion activity product is larger that the logK value.

### Other tabs
On the **Activity correction** tab we select the model equation used by ORCHESTRA to calculate the activities of the species in solution. We will use the Davies model and mark the field **Calculate Ionic Strenghth** which you can check for your self. 
On the **Settings** tab we mark the field **Include SI minerals**. This last item allows us to evaluate which other minerals in the system might become oversaturated, we then may choose to include these in the precipitation reactions as well.

The other tabs we leave as is, but you should take a look at these tabs anyway. On the **Variables** tab the user can set all kinds of constants for the calculations. However, many of these will be "overruled" by us in the Python simulations. The **Adsorption models** tab is used to calculate adsortion of ionic compounds to all kinds of surfaces, lies beyond the scope of this course. The **Predominance Diagram** tab will be used in a later lecture when we discuss redox reactions. Finally the **Output selector** tab is useful to find out what variables are present in our current ORCHESTRA simulation.

### Using Orchestra to run the scenario
It is possible to use the ORCHESTRA GUI to run the simulation, and this has been prepared using the **Input** and **Output** tabs on the right of the window. Please have look at these tabs and they should be rather "self explanatory". 



## Running the Ca-F-Gypsum scenario in a Python notebook
We will use a Python notebook and Orchestra to calculate the trajectory from A via point B to Point C in the example from Appelo & Postma chapter 4.1. We start from the initial situation without any Gypsum and know $\text{Ca}^{+2}$ and $\text{F}^-$ concentrations. Then we add small consecutative amounts of Gypsum until we have a final equilirbium between Gypsum, Fluorite and the solution.

In order to run the Ca-F-Gypsum scenario is a notebook we have to follow a systematic approach which consists of the following standard steps:
1. define the domain for our problem
2. define the primary species which change during our scenario
3. initialize the problem
4. run the problem
5. process the output

### 1. Define the domain
In the example as given by Appelo and Postma, no information is given about the volume of water involved. Concentrations are given in mg/liter. In this case it is a good approach to do the simulation for 1 liter of water (or better for 1 kg of water). 
```{note} Why is 1 kg water a better choice than 1 liter of water?
In your answer, please take the density of the solution in to consideration, and especially how the density may change due changes in the concentrations of solutes.
```
For our simulation we do not require any information about the gas volume or solid volumes.

### 2. Define the primary species
The primary species are as defined above with the ORCHESTRA GUI. Please note that the values required by ORCHESTRA have to be in moles. In order to get moles/liter the volume of water has to be set to 1 liter, and the density of water need to be given as well.

### 3. Initialise the problem 
The initial condition for our problem is the situation without any Gypsum present.
Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry1.inp*, created above with the ORCHESTRA-GUI. After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

The following code shows how to do this.


In [2]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry1.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    InVars = np.array(['Ca+2.tot', 'Fe+2.tot', 
                       'CO2[g].logact',
                       'watervolume',
                       'T'
                       ])
    
    # We select the output from Orchestra we need to use
    OutVars = np.array(['pH', 'H+.con', 'H+.tot', 
                        'CO3-2.tot', 'CO3-2.con', 'CO3-2.logact','CO3-2.diss',
                        'HCO3-.tot', 'HCO3-.con', 'HCO3-.logact',
                        'H2CO3.tot', 'H2CO3.con', 'H2CO3.logact',
                        'Ca+2.tot', 'Ca+2.con', 'Ca+2.logact', 'Ca+2.diss', 
                        'Fe+2.tot', 'Fe+2.con', 'Fe+2.logact', 'Fe+2.diss',
                        'Calcite[s].si', 'Calcite[s].tot', 'Calcite[s].logact',
                        'Siderite[s].si', 'Siderite[s].tot', 'Siderite[s].logact',
                        'T', 'I', 'chargebalance', 'watervolume' ])

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    p = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    p.initialise(InputFile, NoCells, InVars, OutVars)


Reading and expanding calculator new stylechemistry1.inp
Scanning file: chemistry1.inp
Scanning file: objects2026_THe.txt
Scanning file: chemistry1.inp
Scanning file: objects2026_THe.txt
Including file: chemistry1.inp
Scanning file: objects2026_THe.txt
0.022 sec.
	Reading variables .... 0.012 s
testing:
13:Ca+2.tot
27:Fe+2.tot
28:CO2[g].logact
29:watervolume
4:T
16:pH
30:H+.con
31:H+.tot
11:CO3-2.tot
32:CO3-2.con
10:CO3-2.logact
33:CO3-2.diss
34:HCO3-.tot
35:HCO3-.con
36:HCO3-.logact
37:H2CO3.tot
38:H2CO3.con
39:H2CO3.logact
13:Ca+2.tot
40:Ca+2.con
12:Ca+2.logact
41:Ca+2.diss
27:Fe+2.tot
42:Fe+2.con
43:Fe+2.logact
44:Fe+2.diss
45:Calcite[s].si
46:Calcite[s].tot
47:Calcite[s].logact
48:Siderite[s].si
49:Siderite[s].tot
50:Siderite[s].logact
4:T
3:I
8:chargebalance
29:watervolume
Initialise Completed! the following IO parameters will be used:
0 : Node_ID : 0
1 : minTol : 0.001
2 : H2O.logact : 0
3 : I : 0.1
4 : T : 298.15
5 : gas_val_u : 0.1
6 : gas_res : 0
7 : logI : -2
8 : chargebalanc

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

## Three steps
### Step 1 Initial condition $P_{CO2} = 1$ atm

In [3]:
# %%
# Step 1, initial condition.
# InVars need to contain floats
IN = np.array([np.ones_like(InVars)]).astype(float)

IN[0][np.where(InVars == 'Ca+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'Fe+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'CO2[g].logact')] = 0 #log10 (1 atm)
IN[0][np.where(InVars == 'watervolume')] = 1.0 #
IN[0][np.where(InVars== 'T')] = 273.15 + 50 # temperature in K

#print(IN[0])

# Calculate equilibrium conditions for initial situation

OUT = p.set_and_calculate(IN)
Res_Initial = pd.DataFrame(OUT,columns=OutVars)
print(r"Step 1: Initial condition, $P_{CO_2}$[g] = 1 atm")
#print(Res_Initial[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']])

table_md = Res_Initial[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']].to_markdown()

#--- Reporting / Output ---] 
display(Markdown(table_md))


Step 1: Initial condition, $P_{CO_2}$[g] = 1 atm


|    |   Ca+2.tot |   Fe+2.tot |    Ca+2.con |   Fe+2.con |   Calcite[s].si |   Siderite[s].si |   Calcite[s].tot |   Siderite[s].tot |
|---:|-----------:|-----------:|------------:|-----------:|----------------:|-----------------:|-----------------:|------------------:|
|  0 |       0.01 |       0.01 | 1.36588e-19 |          0 |        -29.2028 |                0 |                0 |                 0 |

### Step 2, $P_{CO2}$ = 4.9 atm

In [4]:
# %%
# Step 2, P_CO2 = 4.9 atm
# InVars need to contain floats
#IN = np.array([np.ones_like(InVars)]).astype(float)

IN[0][np.where(InVars == 'Ca+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'Fe+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'CO2[g].logact')] = np.log10(4.9) #log10 (4.9 atm)
IN[0][np.where(InVars == 'watervolume')] = 1.0 #
IN[0][np.where(InVars== 'T')] = 273.15 + 50 # temperature in K

#print(IN[0])
# Calculate equilibrium conditions for initial situation
OUT = p.set_and_calculate(IN)
Res_4_9_atm = pd.DataFrame(OUT,columns=OutVars)
print("Step 2, $P_{CO_2[g]}$ = 4.9 atm")
#print(Res_Initial[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']])

table_md = Res_4_9_atm[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']].to_markdown()

#--- Reporting / Output ---] 
display(Markdown(table_md))

Step 2, $P_{CO_2[g]}$ = 4.9 atm


|    |   Ca+2.tot |   Fe+2.tot |    Ca+2.con |   Fe+2.con |   Calcite[s].si |   Siderite[s].si |   Calcite[s].tot |   Siderite[s].tot |
|---:|-----------:|-----------:|------------:|-----------:|----------------:|-----------------:|-----------------:|------------------:|
|  0 |       0.01 |       0.01 | 1.36588e-19 |          0 |        -29.2028 |                0 |                0 |                 0 |

### Step 3 $P_{CO2} = 10$ atm

In [5]:
# %%
# Step 3, P_CO2 = 10 atm
# InVars need to contain floats
#IN = np.array([np.ones_like(InVars)]).astype(float)

IN[0][np.where(InVars == 'Ca+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'Fe+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'CO2[g].logact')] = np.log10(10) #log10 (10 atm)
IN[0][np.where(InVars == 'watervolume')] = 1.0 #
IN[0][np.where(InVars== 'T')] = 273.15 + 50 # temperature in K

print(IN[0])
# Calculate equilibrium conditions for initial situation
OUT = p.set_and_calculate(IN)
Res_10_atm = pd.DataFrame(OUT,columns=OutVars)
#print(Res_Initial[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']])

table_md = Res_10_atm[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']].to_markdown()

#--- Reporting / Output ---] 
display(Markdown(table_md))

[1.0000e-02 1.0000e-02 1.0000e+00 1.0000e+00 3.2315e+02]


|    |   Ca+2.tot |   Fe+2.tot |    Ca+2.con |   Fe+2.con |   Calcite[s].si |   Siderite[s].si |   Calcite[s].tot |   Siderite[s].tot |
|---:|-----------:|-----------:|------------:|-----------:|----------------:|-----------------:|-----------------:|------------------:|
|  0 |       0.01 |       0.01 | 1.36588e-19 |          0 |        -29.2028 |                0 |                0 |                 0 |

### Step 4: Equilibrate with atmosphere at 10 $^o\text{C}$


In [6]:
# %%
# Step 4, P_CO2 = 420e-6 atm at 283.15 K
# InVars need to contain floats
#IN = np.array([np.ones_like(InVars)]).astype(float)

IN[0][np.where(InVars == 'Ca+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'Fe+2.tot')] = 0.01 #mol/liter
IN[0][np.where(InVars == 'CO2[g].logact')] = np.log10(420e-6) #log10 (10 atm)
IN[0][np.where(InVars == 'watervolume')] = 1.0 #
IN[0][np.where(InVars== 'T')] = 273.15 + 10 # temperature in K

#print(IN[0])
# Calculate equilibrium conditions for initial situation
OUT = p.set_and_calculate(IN)
Res_atm = pd.DataFrame(OUT,columns=OutVars)
print("Step 2, $P_{CO_2[g]}$ = 10 atm")
#print(Res_Initial[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']])

table_md = Res_atm[['Ca+2.tot', 'Fe+2.tot', 'Ca+2.con', 'Fe+2.con', 'Calcite[s].si', 'Siderite[s].si', 'Calcite[s].tot', 'Siderite[s].tot']].to_markdown()

#--- Reporting / Output ---] 
display(Markdown(table_md))

Step 2, $P_{CO_2[g]}$ = 10 atm


|    |   Ca+2.tot |   Fe+2.tot |    Ca+2.con |   Fe+2.con |   Calcite[s].si |   Siderite[s].si |   Calcite[s].tot |   Siderite[s].tot |
|---:|-----------:|-----------:|------------:|-----------:|----------------:|-----------------:|-----------------:|------------------:|
|  0 |       0.01 |       0.01 | 1.50438e-19 |          0 |        -29.7681 |                0 |                0 |                 0 |

## Analysis
The solubility of Calcite and Siderite at 50 $^o$C increases with partial CO2 pressure. At 10 atm we see that the solubility of Calcite exceeds the 0.01 mol/liter of Ca+2 in solution. If the amount of Calcite would be higher, we the concentration of Ca+2 increases. Same story for Fe+2, but the solubility increases less quickly as Ca+2 from Calcite. 